In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import re
import requests
from io import StringIO
from datetime import datetime

In [2]:
# separa el nombre del apodo 
def separar_apodo(df, col_nombre="Winner", nueva_col="WinnerCode"):
    """
    Separa código FIA (3 letras mayúsculas al final) de un nombre completo.
    """

    if col_nombre not in df.columns:
        raise ValueError(f"La columna '{col_nombre}' no existe en el DataFrame.")

    # Extraer código
    df[nueva_col] = df[col_nombre].str.extract(r'([A-Z]{3})$')

    # Limpiar nombre
    df[col_nombre] = (
        df[col_nombre]
        .str.replace(r'[A-Z]{3}$', '', regex=True)
        .str.strip()
    )

    # Reordenar columnas (colocar nueva justo después)
    cols = list(df.columns)
    idx = cols.index(col_nombre)

    cols.remove(nueva_col)
    cols.insert(idx + 1, nueva_col)

    return df[cols]


In [3]:
# funcion para quitar duplicados en nombre grand prix
def limpiar_gp(nombre):
    nombre = str(nombre).strip()
    
    # 1. Caso: texto duplicado exacto (Great BritainGreat Britain)
    mitad = len(nombre) // 2
    if nombre[:mitad] == nombre[mitad:]:
        return nombre[:mitad].strip()
    
    # 2. Caso: algo pegado al final sin espacio (AmericaIndianapolis)
    # buscamos el último punto donde empieza una mayúscula SIN espacio antes
    corte = None
    for i in range(1, len(nombre)):
        if nombre[i].isupper() and nombre[i-1] != ' ':
            corte = i
    
    if corte is not None:
        return nombre[:corte].strip()
    
    # 3. Si no cumple nada de lo anterior, lo dejamos tal cual
    return nombre


In [4]:
headers = {"User-Agent": "Mozilla/5.0"}
base_url = "https://www.formula1.com/en/results/{}/races"
# Fecha actual para determinar último año disponible
now = datetime.now()
last_year = now.year if now.month >= 3 else now.year - 1

dfs = []

for year in range(1950, last_year + 1):
    url = base_url.format(year)
    r = requests.get(url, headers=headers)

    if r.status_code != 200:
        continue

    try:
        df = pd.read_html(StringIO(r.text))[0]
        df["Year"] = year

        # Normalizar nombres de columnas
        df.rename(columns=lambda x: x.strip(), inplace=True)

        if "Winner" in df.columns:
            df["Winner"] = df["Winner"].str.strip()

        if "Team" in df.columns:
            df["Team"] = df["Team"].str.strip()

        dfs.append(df)

    except ValueError:
        pass

# Unir todos los años
f1_races2 = pd.concat(dfs, ignore_index=True)


# Separar nombre y código FIA

if "Winner" in f1_races2.columns:
    
    # Extraer código (3 letras mayúsculas al final)
    f1_races2["WinnerCode"] = f1_races2["Winner"].str.extract(r'([A-Z]{3})$')
    
    # Limpiar nombre quitando el código
    f1_races2["Winner"] = (
        f1_races2["Winner"]
        .str.replace(r'[A-Z]{3}$', '', regex=True)
        .str.strip()
    )

    # Reordenar columnas: colocar WinnerCode justo después de Winner
    cols = list(f1_races2.columns)
    winner_index = cols.index("Winner")
    
    if "WinnerCode" in cols:
        cols.remove("WinnerCode")
        cols.insert(winner_index + 1, "WinnerCode")
    
    f1_races = f1_races2[cols]

# Limpiar columna "Grand Prix"
if "Grand Prix" in f1_races.columns:
    f1_races["Grand Prix"] = (
        f1_races["Grand Prix"]
        .str.replace(r'^Flag of ', '', regex=True)
        .str.strip()
    )

# funcion para quitar duplicados en nombre grand prix
f1_races['Grand Prix'] = f1_races['Grand Prix'].apply(limpiar_gp)

# ver primeras filas
f1_races

,Grand Prix,Date,Winner,WinnerCode,Team,Laps,Time,Year
0,Great Britain,13 May,Nino Farina,FAR,Alfa Romeo,70.0,2:13:23.600,1950
1,Monaco,21 May,Juan Manuel Fangio,FAN,Alfa Romeo,100.0,3:13:18.700,1950
2,United States of America,30 May,Johnnie Parsons,PAR,Kurtis Kraft Offenhauser,138.0,2:46:55.970,1950
3,Switzerland,04 Jun,Nino Farina,FAR,Alfa Romeo,42.0,2:02:53.700,1950
4,Belgium,18 Jun,Juan Manuel Fangio,FAN,Alfa Romeo,35.0,2:47:26.000,1950
...,...,...,...,...,...,...,...,...
1147,Mexico,26 Oct,Lando Norris,NOR,McLaren,71.0,1:37:58.574,2025
1148,Brazil,09 Nov,Lando Norris,NOR,McLaren,71.0,1:32:01.596,2025
1149,United States of America,22 Nov,Max Verstappen,VER,Red Bull Racing,50.0,1:21:08.429,2025
1150,Qatar,30 Nov,Max Verstappen,VER,Red Bull Racing,57.0,1:24:38.241,2025


In [5]:

headers = {"User-Agent": "Mozilla/5.0"}
base_url = "https://www.formula1.com/en/results/{}/drivers"

# fecha actual para determinar el último año
now = datetime.now()
last_year = now.year if now.month >= 3 else now.year - 1

dfs = []

for year in range(1950, last_year + 1):
    url = base_url.format(year)
    r = requests.get(url, headers=headers)
    if r.status_code != 200:
        continue

    try:
        df = pd.read_html(StringIO(r.text))[0]
        df["Year"] = year
        # Normalizar nombres de columna según el formato de drivers
        df.rename(columns=lambda x: x.strip(), inplace=True)
        if "Driver" in df.columns:
            df["Driver"] = df["Driver"].str.strip()
        if "Team" in df.columns:
            df["Team"] = df["Team"].str.strip()
        dfs.append(df)
    except ValueError:
        pass

# unir todos los años
f1_drivers = pd.concat(dfs, ignore_index=True)

# Extraer el código FIA (3 letras mayúsculas al final)
f1_drivers['DriverCode'] = f1_drivers['Driver'].str.extract(r'([A-Z]{3})$')

# Limpiar el nombre quitando las 3 letras finales
f1_drivers['Driver'] = f1_drivers['Driver'].str.replace(r'[A-Z]{3}$', '', regex=True).str.strip()
#f1_drivers

cols = list(f1_drivers.columns)

# Sacar DriverCode de donde esté
cols.remove('DriverCode')

# Insertarla justo después de Driver
driver_index = cols.index('Driver') + 1
cols.insert(driver_index, 'DriverCode')

# Reordenar el dataframe
f1_drivers = f1_drivers[cols]


# ver primeras filas
f1_drivers



,Pos.,Driver,DriverCode,Nationality,Team,Pts.,Year
0,1,Nino Farina,FAR,ITA,Alfa Romeo,30.0,1950
1,2,Juan Manuel Fangio,FAN,ARG,Alfa Romeo,27.0,1950
2,3,Luigi Fagioli,FAG,ITA,Alfa Romeo,24.0,1950
3,4,Louis Rosier,ROS,FRA,Talbot-Lago,13.0,1950
4,5,Alberto Ascari,ASC,ITA,NaN,11.0,1950
...,...,...,...,...,...,...,...
1651,17,Yuki Tsunoda,TSU,JPN,Red Bull Racing,33.0,2025
1652,18,Pierre Gasly,GAS,FRA,Alpine,22.0,2025
1653,19,Gabriel Bortoleto,BOR,BRA,Kick Sauber,19.0,2025
1654,20,Franco Colapinto,COL,ARG,Alpine,0.0,2025


In [6]:
headers = {"User-Agent": "Mozilla/5.0"}
base_url = "https://www.formula1.com/en/results/{}/team"

now = datetime.now()
last_year = now.year if now.month >= 3 else now.year - 1

dfs = []

for year in range(1950, last_year + 1):
    url = base_url.format(year)
    r = requests.get(url, headers=headers)
    if r.status_code != 200:
        continue
    try:
        df = pd.read_html(StringIO(r.text))[0]
        df["Year"] = year
        # Limpiar nombres de equipos si existe la columna Team
        if "Team" in df.columns:
            df["Team"] = df["Team"].str.strip()
        dfs.append(df)
    except ValueError:
        continue

f1_teams = pd.concat(dfs, ignore_index=True)


# print
f1_teams

,Pos.,Team,Pts.,Year
0,1,Vanwall,48.0,1958
1,2,Ferrari,40.0,1958
2,3,Cooper Climax,31.0,1958
3,4,BRM,18.0,1958
4,5,Maserati,6.0,1958
...,...,...,...,...
700,6,Racing Bulls,92.0,2025
701,7,Aston Martin,89.0,2025
702,8,Haas F1 Team,79.0,2025
703,9,Kick Sauber,70.0,2025


In [7]:

headers = {"User-Agent": "Mozilla/5.0"}
base_url = "https://www.formula1.com/en/results/{}/awards/pole-positions"

now = datetime.now()
last_year = now.year if now.month >= 3 else now.year - 1

dfs = []

for year in range(1950, last_year + 1):
    url = base_url.format(year)
    r = requests.get(url, headers=headers)
    if r.status_code != 200:
        continue
    try:
        df = pd.read_html(StringIO(r.text))[0]
        df["Year"] = year
        df["Grand Prix"] = df["Grand Prix"].str.replace(r"Flag of .*?", "", regex=True)
        dfs.append(df)
    except ValueError:
        continue

f1_pole = pd.concat(dfs, ignore_index=True)

# funcion para quitar duplicados en nombre grand prix
f1_pole['Grand Prix'] = f1_pole['Grand Prix'].apply(limpiar_gp)

# funcion para separar el nombre del apodo 
f1_pole = separar_apodo(f1_pole, "Winner", "WinnerCode")


# print
f1_pole

,Grand Prix,Winner,WinnerCode,Time,Year
0,Great Britain,Nino Farina,FAR,1:50.800,1950
1,Monaco,Juan Manuel Fangio,FAN,1:50.200,1950
2,United States of America,Walt Faulkner,FAU,4:27.970,1950
3,Switzerland,Juan Manuel Fangio,FAN,2:42.100,1950
4,Belgium,Nino Farina,FAR,4:37.000,1950
...,...,...,...,...,...
1144,Mexico,Lando Norris,NOR,1:15.586,2025
1145,Brazil,Lando Norris,NOR,1:09.511,2025
1146,United States of America,Lando Norris,NOR,1:47.934,2025
1147,Qatar,Oscar Piastri,PIA,1:19.387,2025


In [8]:

headers = {"User-Agent": "Mozilla/5.0"}
base_url = "https://www.formula1.com/en/results/{}/awards/fastest-laps"

now = datetime.now()
last_year = now.year if now.month >= 3 else now.year - 1

dfs = []

for year in range(1950, last_year + 1):
    url = base_url.format(year)
    r = requests.get(url, headers=headers)
    if r.status_code != 200:
        continue
    try:
        df = pd.read_html(StringIO(r.text))[0]
        df["Year"] = year
        df["Grand Prix"] = df["Grand Prix"].str.replace(r"Flag of .*?", "", regex=True)
        dfs.append(df)
    except ValueError:
        continue

f1_fastlaps = pd.concat(dfs, ignore_index=True)

# funcion para quitar duplicados en nombre grand prix
f1_fastlaps['Grand Prix'] = f1_fastlaps['Grand Prix'].apply(limpiar_gp)

# funcion para separar el nombre del apodo 
f1_fastlaps = separar_apodo(f1_fastlaps, "Winner", "WinnerCode")

# print faster laps
f1_fastlaps

,Grand Prix,Winner,WinnerCode,Time,Year
0,Great Britain,Nino Farina,FAR,1:50.600,1950
1,Monaco,Juan Manuel Fangio,FAN,1:51.000,1950
2,United States of America,Johnnie Parsons,PAR,NaN,1950
3,Switzerland,Nino Farina,FAR,2:41.600,1950
4,Belgium,Nino Farina,FAR,4:34.100,1950
...,...,...,...,...,...
1143,Mexico,George Russell,RUS,1:20.052,2025
1144,Brazil,Alexander Albon,ALB,1:12.400,2025
1145,United States of America,Max Verstappen,VER,1:33.365,2025
1146,Qatar,Oscar Piastri,PIA,1:22.996,2025


In [9]:
headers = {"User-Agent": "Mozilla/5.0"}
base_url = "https://www.formula1.com/en/results/{}/awards/fastest-pit-stops"

now = datetime.now()
last_year = now.year if now.month >= 3 else now.year - 1

dfs = []

for year in range(1950, last_year + 1):
    url = base_url.format(year)
    r = requests.get(url, headers=headers)

    if r.status_code != 200:
        continue

    try:
        df = pd.read_html(StringIO(r.text))[0]
        df["Year"] = year

        # Limpiar "Grand Prix"
        df["Grand Prix"] = (
            df["Grand Prix"]
            .str.replace(r"Flag of .*?", "", regex=True)
            .str.strip()
        )

        dfs.append(df)

    except ValueError:
        continue

# Unir todos los años
f1_fast_pit = pd.concat(dfs, ignore_index=True)


# funcion para quitar duplicados en nombre grand prix
f1_fast_pit['Grand Prix'] = f1_fast_pit['Grand Prix'].apply(limpiar_gp)


# ==============================
# Filtrar solo tiempos que terminan en "s"
# ==============================
if "Time" in f1_fast_pit.columns:
    f1_fast_pit = f1_fast_pit[
        f1_fast_pit["Time"].astype(str).str.strip().str.endswith("s")
    ]

# Ver resultado
#f1_fast_pit.count()
f1_fast_pit


,Grand Prix,Winner,Time,Year
937,Germany,Williams,2.09s,2016
938,Australia,Williams,2.34s,2017
939,People’s Republic of China,Ferrari,2.42s,2017
940,Bahrain,Williams,2.34s,2017
941,Russia,Red Bull Racing,2.33s,2017
...,...,...,...,...
1127,Mexico,McLaren,2.10s,2025
1128,Brazil,Red Bull Racing,1.95s,2025
1129,United States of America,Red Bull Racing,1.99s,2025
1130,Qatar,McLaren,2.08s,2025


In [10]:
with pd.ExcelWriter("F1.xlsx") as writer:
    if not f1_races.empty:
        f1_races.to_excel(writer, sheet_name="Racers", index=False)
    if not f1_drivers.empty:
        f1_drivers.to_excel(writer, sheet_name="Drivers", index=False)
    if not f1_teams.empty:
        f1_teams.to_excel(writer, sheet_name="Teams", index=False)
    if not f1_fastlaps.empty:
        f1_fastlaps.to_excel(writer, sheet_name="FastLaps", index=False)
    if not f1_pole.empty:
        f1_pole.to_excel(writer, sheet_name="Pole", index=False)
    if not f1_fast_pit.empty:
        f1_fast_pit.to_excel(writer, sheet_name="FastPit", index=False)
        
print("Excel generado con éxito: F1.xlsx")

Excel generado con éxito: F1.xlsx


### Leer los datos del excel

In [ ]:
#Capturar archivo de datos
path = r'C:\turutadecarpetas\'
filef1 = 'F1.xlsx'

#Path
ruta_f1 = os.path.join(path, filef1)

df1_F1 = pd.read_excel(ruta_f1, sheet_name='Racers')
df1_F1
#df.count()


,Grand Prix,Date,Winner,WinnerCode,Team,Laps,Time,Year
0,Great Britain,13 May,Nino Farina,FAR,Alfa Romeo,70.0,2:13:23.600,1950
1,Monaco,21 May,Juan Manuel Fangio,FAN,Alfa Romeo,100.0,3:13:18.700,1950
2,United States of America,30 May,Johnnie Parsons,PAR,Kurtis Kraft Offenhauser,138.0,2:46:55.970,1950
3,Switzerland,04 Jun,Nino Farina,FAR,Alfa Romeo,42.0,2:02:53.700,1950
4,Belgium,18 Jun,Juan Manuel Fangio,FAN,Alfa Romeo,35.0,2:47:26.000,1950
...,...,...,...,...,...,...,...,...
1147,Mexico,26 Oct,Lando Norris,NOR,McLaren,71.0,1:37:58.574,2025
1148,Brazil,09 Nov,Lando Norris,NOR,McLaren,71.0,1:32:01.596,2025
1149,United States of America,22 Nov,Max Verstappen,VER,Red Bull Racing,50.0,1:21:08.429,2025
1150,Qatar,30 Nov,Max Verstappen,VER,Red Bull Racing,57.0,1:24:38.241,2025


In [12]:
df2_F1 = pd.read_excel(ruta_f1, sheet_name='Drivers')
df2_F1
#df2_F1.count()

,Pos.,Driver,DriverCode,Nationality,Team,Pts.,Year
0,1,Nino Farina,FAR,ITA,Alfa Romeo,30.0,1950
1,2,Juan Manuel Fangio,FAN,ARG,Alfa Romeo,27.0,1950
2,3,Luigi Fagioli,FAG,ITA,Alfa Romeo,24.0,1950
3,4,Louis Rosier,ROS,FRA,Talbot-Lago,13.0,1950
4,5,Alberto Ascari,ASC,ITA,NaN,11.0,1950
...,...,...,...,...,...,...,...
1651,17,Yuki Tsunoda,TSU,JPN,Red Bull Racing,33.0,2025
1652,18,Pierre Gasly,GAS,FRA,Alpine,22.0,2025
1653,19,Gabriel Bortoleto,BOR,BRA,Kick Sauber,19.0,2025
1654,20,Franco Colapinto,COL,ARG,Alpine,0.0,2025


In [13]:
df3_F1 = pd.read_excel(ruta_f1, sheet_name='Teams')
df3_F1
#df3_F1.count()

,Pos.,Team,Pts.,Year
0,1,Vanwall,48.0,1958
1,2,Ferrari,40.0,1958
2,3,Cooper Climax,31.0,1958
3,4,BRM,18.0,1958
4,5,Maserati,6.0,1958
...,...,...,...,...
700,6,Racing Bulls,92.0,2025
701,7,Aston Martin,89.0,2025
702,8,Haas F1 Team,79.0,2025
703,9,Kick Sauber,70.0,2025


In [14]:
df_merged = pd.merge(
    df1_F1,
    df2_F1,
    left_on=['Winner', 'Year'],
    right_on=['Driver', 'Year'],
    how='left'
)

df_merged


,Grand Prix,Date,Winner,WinnerCode,Team_x,Laps,Time,Year,Pos.,Driver,DriverCode,Nationality,Team_y,Pts.
0,Great Britain,13 May,Nino Farina,FAR,Alfa Romeo,70.0,2:13:23.600,1950,1,Nino Farina,FAR,ITA,Alfa Romeo,30.0
1,Monaco,21 May,Juan Manuel Fangio,FAN,Alfa Romeo,100.0,3:13:18.700,1950,2,Juan Manuel Fangio,FAN,ARG,Alfa Romeo,27.0
2,United States of America,30 May,Johnnie Parsons,PAR,Kurtis Kraft Offenhauser,138.0,2:46:55.970,1950,6,Johnnie Parsons,PAR,USA,Kurtis Kraft Offenhauser,9.0
3,Switzerland,04 Jun,Nino Farina,FAR,Alfa Romeo,42.0,2:02:53.700,1950,1,Nino Farina,FAR,ITA,Alfa Romeo,30.0
4,Belgium,18 Jun,Juan Manuel Fangio,FAN,Alfa Romeo,35.0,2:47:26.000,1950,2,Juan Manuel Fangio,FAN,ARG,Alfa Romeo,27.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1147,Mexico,26 Oct,Lando Norris,NOR,McLaren,71.0,1:37:58.574,2025,1,Lando Norris,NOR,GBR,McLaren,423.0
1148,Brazil,09 Nov,Lando Norris,NOR,McLaren,71.0,1:32:01.596,2025,1,Lando Norris,NOR,GBR,McLaren,423.0
1149,United States of America,22 Nov,Max Verstappen,VER,Red Bull Racing,50.0,1:21:08.429,2025,2,Max Verstappen,VER,NED,Red Bull Racing,421.0
1150,Qatar,30 Nov,Max Verstappen,VER,Red Bull Racing,57.0,1:24:38.241,2025,2,Max Verstappen,VER,NED,Red Bull Racing,421.0


In [15]:
df_full = pd.merge(
    df2_F1,
    df1_F1,
    left_on='Year',
    right_on='Year',
    how='outer'
)
df_full

,Pos.,Driver,DriverCode,Nationality,Team_x,Pts.,Year,Grand Prix,Date,Winner,WinnerCode,Team_y,Laps,Time
0,1,Nino Farina,FAR,ITA,Alfa Romeo,30.0,1950,Great Britain,13 May,Nino Farina,FAR,Alfa Romeo,70.0,2:13:23.600
1,1,Nino Farina,FAR,ITA,Alfa Romeo,30.0,1950,Monaco,21 May,Juan Manuel Fangio,FAN,Alfa Romeo,100.0,3:13:18.700
2,1,Nino Farina,FAR,ITA,Alfa Romeo,30.0,1950,United States of America,30 May,Johnnie Parsons,PAR,Kurtis Kraft Offenhauser,138.0,2:46:55.970
3,1,Nino Farina,FAR,ITA,Alfa Romeo,30.0,1950,Switzerland,04 Jun,Nino Farina,FAR,Alfa Romeo,42.0,2:02:53.700
4,1,Nino Farina,FAR,ITA,Alfa Romeo,30.0,1950,Belgium,18 Jun,Juan Manuel Fangio,FAN,Alfa Romeo,35.0,2:47:26.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25261,21,Jack Doohan,DOO,AUS,Alpine,0.0,2025,Mexico,26 Oct,Lando Norris,NOR,McLaren,71.0,1:37:58.574
25262,21,Jack Doohan,DOO,AUS,Alpine,0.0,2025,Brazil,09 Nov,Lando Norris,NOR,McLaren,71.0,1:32:01.596
25263,21,Jack Doohan,DOO,AUS,Alpine,0.0,2025,United States of America,22 Nov,Max Verstappen,VER,Red Bull Racing,50.0,1:21:08.429
25264,21,Jack Doohan,DOO,AUS,Alpine,0.0,2025,Qatar,30 Nov,Max Verstappen,VER,Red Bull Racing,57.0,1:24:38.241
